# Week 3 — Customer Segmentation with RFM & K-Means Clustering

This notebook walks through the complete unsupervised-learning pipeline implemented in src/: RFM (Recency / Frequency / Monetary) feature engineering on the UCI Online Retail Dataset, log1p + StandardScaler transforms, k-selection via Elbow + Silhouette analysis, K-Means clustering, internal evaluation metrics, PCA 2D visualization, and evidence-based per-cluster descriptions. All numbers shown below are computed from the actual loaded dataset.

In [ ]:
# --- Step 1: Project setup & library imports ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
assert (PROJECT_ROOT / "src" / "main.py").exists(), (
    "Run this notebook from the online-retail-clustering/ project root directory"
)

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:,.3f}".format)

from src.config import RANDOM_SEED
from src.data_loading import load_excel_dataset
from src.data_preparation import prepare_all, get_positive_sales_with_customer_id
from src.feature_engineering import (
    compute_rfm,
    inspect_rfm_distributions,
    apply_log1p_transformation,
    scale_rfm_features,
)
from src.clustering import (
    find_optimal_k,
    run_kmeans,
    apply_pca_for_visualization,
    compute_cluster_profiles,
)
from src.evaluation import (
    compute_clustering_metrics,
    rank_clusters_by_profit,
    describe_cluster,
)
from src.visualization import run_all_visualizations

print("Imports OK.")
print(f"  PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"  pandas {pd.__version__}  |  numpy {np.__version__}  |  random_seed = {RANDOM_SEED}")

In [ ]:
# --- Step 2: Load the raw UCI Online Retail Excel dataset ---
df_raw = None
try:
    df_raw = load_excel_dataset()
    print(f"Dataset loaded OK:  {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
    print(f"Columns: {list(df_raw.columns)}")
    print(f"Date range: {df_raw['InvoiceDate'].min()}  ->  {df_raw['InvoiceDate'].max()}")
    print(f"Unique customers (incl. nulls):  {df_raw['CustomerID'].nunique():,}")
    null_cid = int(df_raw['CustomerID'].isna().sum())
    print(f"Rows with NULL CustomerID:  {null_cid:,}  ({null_cid/len(df_raw)*100:.2f}% of all lines)")
except FileNotFoundError as e:
    df_raw = None
    print("Dataset not found.")
    print("   Please download the UCI Online Retail Excel file and place it at:")
    print(f"   {PROJECT_ROOT / 'data' / 'Online+Retail.xlsx'}")
    print("   Source URL: https://archive.ics.uci.edu/dataset/352/online+retail")
except Exception as e:
    df_raw = None
    print(f"Failed to load dataset: {type(e).__name__}: {e}")

In [ ]:
# --- Step 3: Feature engineering + analytical-dataset construction ---
datasets = None
df_pos = None
df_pos_cid = None

if df_raw is not None:
    datasets = prepare_all(df_raw)
    df_full = datasets["full"]
    df_valid_price = datasets["valid_price"]
    df_pos = datasets["positive_sales"]
    df_pos_cid = get_positive_sales_with_customer_id(df_pos)
    print("Row counts after data preparation:")
    print(f"  full (after dedup):            {len(df_full):>10,}")
    print(f"  valid_price (UnitPrice > 0):   {len(df_valid_price):>10,}")
    print(f"  positive_sales (non-cancel, Qty>0, Price>0):  {len(df_pos):>10,}")
    print(f"  positive_sales WITH CustomerID (for RFM):     {len(df_pos_cid):>10,}")
    print()
    print(f"Distinct customers with valid IDs:  {df_pos_cid['CustomerID'].nunique():,}")
    print(f"Columns engineered: {sorted(set(df_full.columns) - set(df_raw.columns))}")
else:
    print("Skipping data preparation - dataset was not loaded in the previous cell.")

## RFM Feature Engineering

For every identifiable customer we compute three RFM metrics, using reference_date = max(InvoiceDate) + 1 day as the analysis snapshot:

| Metric | Definition |
|---|---|
| **Recency**   | Days between the reference date and the customer most recent invoice date |
| **Frequency** | Number of distinct invoices (orders) placed by the customer |
| **Monetary**  | Sum of TotalAmount (Quantity x UnitPrice) across all positive-sales lines for the customer |

In [ ]:
# --- Step 4: Compute per-customer RFM and view distributions ---
rfm_df = None
rfm_distributions = None

if df_pos_cid is not None:
    rfm_df = compute_rfm(df_pos_cid)
    rfm_distributions = inspect_rfm_distributions(rfm_df)
    print(f"RFM computed for {len(rfm_df):,} unique customers")
    print()
    print("First 5 rows of RFM table (CustomerID keyed):")
    display(rfm_df.head())
    print()
    print("RFM descriptive statistics (describe + skew + kurtosis):")
    rfm_stats_rows = []
    for col in ["Recency", "Frequency", "Monetary"]:
        row = rfm_distributions[col].copy()
        row.index = [col]
        rfm_stats_rows.append(row)
    rfm_stats_df = pd.concat(rfm_stats_rows, axis=0)
    display(rfm_stats_df)

In [ ]:
# --- Step 5: log1p transform + StandardScaler ---
rfm_df_log = None
scaled_features = None
scaler = None
FEATURE_COLS_LOG = ["Log_Recency", "Log_Frequency", "Log_Monetary"]

if rfm_df is not None:
    rfm_df_log = apply_log1p_transformation(rfm_df, columns=["Recency", "Frequency", "Monetary"])
    scaled_features, scaler = scale_rfm_features(rfm_df_log, feature_cols=FEATURE_COLS_LOG)
    print("Transform & scale summary:")
    print(f"  rfm_df_log shape:        {rfm_df_log.shape[0]} rows  x  {rfm_df_log.shape[1]} cols")
    print(f"  scaled_features shape:   {scaled_features.shape}")
    print(f"  Scaler fitted means:     {scaler.mean_.round(4)}")
    print(f"  Scaler fitted stds:      {scaler.scale_.round(4)}")
    print()
    print("Skewness comparison (raw vs log1p):")
    skew_raw = rfm_df[["Recency", "Frequency", "Monetary"]].skew().to_frame("raw_skew")
    skew_log = rfm_df_log[FEATURE_COLS_LOG].skew().to_frame("log1p_skew")
    skew_log.index = ["Recency", "Frequency", "Monetary"]
    skew_compare = pd.concat([skew_raw, skew_log], axis=1)
    skew_compare["skew_reduction_pct"] = (
        (skew_compare["raw_skew"].abs() - skew_compare["log1p_skew"].abs())
        / skew_compare["raw_skew"].abs() * 100
    )
    display(skew_compare.round(3))
else:
    print("Skipping transforms - rfm_df not available.")

## Optimal K Selection - Elbow & Silhouette

K-Means is fit for every integer k in [2, 10] (random_state=42, n_init=10). For each k we record Inertia (within-cluster sum of squares - the Elbow plot) and the mean Silhouette coefficient (higher = better separation and cohesion). The final k is chosen by taking the k with the maximum Silhouette score, with the Elbow plot used as corroborating visual evidence.

In [ ]:
# --- Step 6: Sweep k = 2..10 and find the best k ---
k_metrics_df = None
best_k = None
best_silhouette = None

if scaled_features is not None:
    k_metrics_df = find_optimal_k(scaled_features)
    best_idx = k_metrics_df["silhouette_score"].idxmax()
    best_k = int(k_metrics_df.loc[best_idx, "k"])
    best_silhouette = float(k_metrics_df.loc[best_idx, "silhouette_score"])
    print("K-selection sweep results (k from 2 to 10):")
    display(k_metrics_df.style.format({
        "k": "{:.0f}",
        "inertia": "{:,.2f}",
        "silhouette_score": "{:.4f}",
    }).highlight_max(subset=["silhouette_score"], color="#d4edda"))
    print()
    print(f"Best k by silhouette score  =  k = {best_k}")
    print(f"    Silhouette score at k={best_k}  =  {best_silhouette:.4f}")
    print(f"    Inertia at k={best_k}             =  {k_metrics_df.loc[best_idx, 'inertia']:,.2f}")
else:
    print("Skipping k-sweep - scaled_features not available.")

## K-Means Clustering & Evaluation

With the optimal k chosen we re-fit K-Means (random_state=42, n_init=10), evaluate internal quality via three standard unsupervised metrics, then compute cluster profiles and evidence-based natural-language descriptions.

In [ ]:
# --- Step 7: Run final K-Means, attach labels, and compute metrics ---
labels = None
kmeans_model = None
clustering_metrics = None
rfm_labeled = None

if scaled_features is not None and best_k is not None:
    labels, kmeans_model = run_kmeans(scaled_features, k=best_k)
    clustering_metrics = compute_clustering_metrics(scaled_features, labels)
    rfm_labeled = rfm_df_log.copy()
    rfm_labeled["Cluster"] = labels.astype(int)
    print(f"K-Means final fit  (k={best_k}, random_state={RANDOM_SEED}, n_init=10)")
    print(f"  Number of labeled customers:  {len(labels):,}")
    print(f"  Final inertia (WCSS):         {kmeans_model.inertia_:,.2f}")
    print()
    print("Internal cluster-evaluation metrics:")
    for k_m, v in clustering_metrics.items():
        print(f"  {k_m:>28s}  =  {v:,.4f}")
    print()
    print("Cluster size distribution (customer count + % of base):")
    size_series = pd.Series(labels, name="Cluster").value_counts().sort_index()
    size_df = pd.DataFrame({
        "cluster": size_series.index.astype(int),
        "customer_count": size_series.values,
        "pct_of_customers": (size_series.values / len(labels) * 100).round(2),
    }).set_index("cluster")
    display(size_df)
else:
    print("Skipping K-Means fit - inputs not available.")

In [ ]:
# --- Step 8: Cluster profiles (aggregate RFM means, revenue share, etc.) ---
cluster_profiles = None
ranked_profiles = None

if rfm_labeled is not None:
    cluster_profiles = compute_cluster_profiles(rfm_labeled, cluster_col="Cluster")
    ranked_profiles = rank_clusters_by_profit(cluster_profiles, by_total=True)
    print("Cluster profiles (sorted by cluster id):")
    display(cluster_profiles.style.format({
        "cluster": "{:.0f}",
        "customer_count": "{:,.0f}",
        "pct_of_customers": "{:.2f}%",
        "mean_recency": "{:.1f}",
        "median_recency": "{:.1f}",
        "mean_frequency": "{:.2f}",
        "median_frequency": "{:.2f}",
        "mean_monetary": "${:,.2f}",
        "median_monetary": "${:,.2f}",
        "total_monetary": "${:,.2f}",
        "pct_of_total_revenue": "{:.2f}%",
    }))
    print()
    c0_total = ranked_profiles.iloc[0]["total_monetary"]
    c0_pct = ranked_profiles.iloc[0]["pct_of_total_revenue"]
    c0_id = int(ranked_profiles.iloc[0]["cluster"])
    print(f"Top cluster by total revenue:  Cluster #{c0_id}"
          f"  with ${c0_total:,.2f}"
          f"  ({c0_pct:.1f}% of all captured revenue).")
else:
    print("Skipping cluster profiles - rfm_labeled not available.")

In [ ]:
# --- Step 9: Evidence-based natural-language descriptions per cluster ---
overall_rfm_stats = {}

if cluster_profiles is not None and rfm_df is not None:
    overall_rfm_stats = {
        "Recency":   float(rfm_df["Recency"].mean()),
        "Frequency": float(rfm_df["Frequency"].mean()),
        "Monetary":  float(rfm_df["Monetary"].mean()),
    }
    print("Overall (dataset-wide) RFM means - used as the comparison baseline:")
    print(f"  Recency   = {overall_rfm_stats['Recency']:.2f} days")
    print(f"  Frequency = {overall_rfm_stats['Frequency']:.2f} orders")
    print(f"  Monetary  = ${overall_rfm_stats['Monetary']:,.2f}")
    print()
    print("=" * 72)
    print("Cluster descriptions (evidence-based, R/F/M vs overall means):")
    print("=" * 72)
    for _, row in cluster_profiles.iterrows():
        desc = describe_cluster(row, overall_rfm_stats)
        print(desc)
        print("-" * 72)
else:
    print("Skipping descriptions - cluster_profiles or rfm_df not available.")

## Visualization

The cell below runs all visualization routines and writes 9+ PNG figures to outputs/figures/ (200 DPI), then shows a sample inline. Figures generated:

1. Raw-scale RFM distributions
2. Log1p-transformed RFM distributions
3. Elbow (Inertia vs k)
4. Silhouette score vs k (best k marked)
5. Cluster size bar chart (counts + percentages)
6. PCA 2D scatter (PC1 vs PC2, coloured by cluster + centroids)
7. Standardized RFM profile heatmap
8. Normalized RFM radar charts (one subplot per cluster)
9. Average monetary spend bar comparison across clusters

In [ ]:
# --- Step 10: Run full visualization pipeline + display sample plots ---
pca_df = None
pca_evr = None
pca_model = None
viz_paths = None

if df_raw is not None and rfm_labeled is not None and k_metrics_df is not None:
    pca_df, pca_evr, pca_model = apply_pca_for_visualization(scaled_features, n_components=2)
    profile_cols_rfm = ["mean_recency", "mean_frequency", "mean_monetary"]
    profiles_scaled_raw = cluster_profiles[profile_cols_rfm].copy()
    profiles_scaled_raw.columns = ["Recency", "Frequency", "Monetary"]
    profiles_scaled_z = (profiles_scaled_raw - profiles_scaled_raw.mean()) / profiles_scaled_raw.std(ddof=0)
    profiles_scaled_z.insert(0, "Cluster", cluster_profiles["cluster"].astype(int).values)
    profiles_for_radar = cluster_profiles[["cluster", "mean_recency", "mean_frequency", "mean_monetary"]].copy()
    profiles_for_radar.columns = ["Cluster", "recency", "frequency", "monetary"]
    viz_paths = run_all_visualizations(
        rfm_df=rfm_df_log.rename(columns={"Recency": "recency", "Frequency": "frequency", "Monetary": "monetary"}),
        rfm_df_log=rfm_df_log,
        k_metrics_df=k_metrics_df,
        labels=rfm_labeled,
        pca_df=pca_df,
        cluster_profiles=profiles_for_radar,
        cluster_profiles_scaled=profiles_scaled_z,
    )
    print(f"PCA explained variance:  PC1 = {pca_evr[0]*100:.2f}%   |   PC2 = {pca_evr[1]*100:.2f}%")
    print(f"Total variance captured by 2 PCs:  {sum(pca_evr)*100:.2f}%")
    print()
    print("Figures written to outputs/figures/:")
    for name, path in viz_paths.items():
        print(f"  {name:>22s}  ->  {path.name}")
    print()
    get_ipython().run_line_magic("matplotlib", "inline")
    from IPython.display import Image, display
    for fig_name in ["elbow", "silhouette", "pca_clusters", "profile_radar"]:
        if fig_name in viz_paths and viz_paths[fig_name].exists():
            print(f"\n=== {fig_name.upper()} === {viz_paths[fig_name].name}")
            display(Image(filename=str(viz_paths[fig_name]), width=900))
else:
    print("Skipping visualization - prerequisite data (dataset, labels, k-metrics) not available.")

## Report & Outputs

See report/Week_3_Unsupervised_Learning_and_Clustering_Report.docx and the outputs/ directory for the full set of generated artifacts:

- **outputs/figures/** - 9+ publication-quality PNG plots (200 DPI)
- **outputs/tables/** - All intermediate and final tables as CSV (RFM raw + log, k-metrics, cluster labels, profiles, PCA coordinates) and JSON (clustering metrics, cluster descriptions)
- **report/Week_3_Unsupervised_Learning_and_Clustering_Report.docx** - Complete course report with methodology, results, figures, tables, limitations, and references.

To reproduce everything identically from the command line without using this notebook:

```bash
# from online-retail-clustering/ project root, after placing Online+Retail.xlsx in data/
python -m src.main
pytest tests/ -v
```